# Evaluation Walkthrough on the Churn Dataset

**Week 2 · Lesson 11** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week2/lectures/11_evaluation_walkthrough_churn.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Adapted from ISLP Ch. 5.3 Lab where relevant.*

## Setup

Same boilerplate as every notebook this term — see Lesson 8 if this is your
first time seeing it.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Business question:** which customers are likely to cancel next billing
cycle, and which of those should get a retention offer?

**Why it matters:** missing an at-risk customer is lost revenue; offering
everyone a discount wastes budget on customers who were never leaving —
the same cost asymmetry Week 1 Lesson 8 opened with. That notebook ran the
*whole pipeline once* on a black-box model and judged it with plain
accuracy, flagged there as a first pass. This notebook is the payoff of
this week: the same dataset, the same model family, now judged with the
metrics, baselines, and selection discipline built across Lessons 1–10 —
because on a dataset where only ~27% of customers churn, "accuracy" alone
was never telling the whole story.

## 2. Data

Telco Customer Churn — one row per customer, `Churn` the binary label.
Loaded via `load_churn()` (`gbus564_coursework.utils.course_data`), the same loader Week 1 Lesson 8 used, so this notebook picks up exactly where that one left off.

In [ ]:
from gbus564_coursework.utils.course_data import load_churn

df = load_churn()
print(f"positive (churn) rate: {df['Churn'].mean():.1%}")
df.head()

### Data Prep

**Hold out the test set, then define the encoding** — the same two steps
as Week 1 Lesson 8:

- **Split first.** 20% of customers are set aside and never touched until
  Section 4. `stratify=y` keeps the ~27% churn rate the same in both halves.
- **Define the encoding — don't apply it yet.** The one-hot encoder learns
  which categories exist from whatever it's fitted on, so it must only see
  training rows. This cell only *defines* it; Section 3 fits it inside the
  model's pipeline — and inside every CV fold — on training data alone.

**Define in Data, fit in Model, on training data only.**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

X, y = df.drop(columns="Churn"), df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

categorical_cols = X.select_dtypes(exclude="number").columns
preprocess = ColumnTransformer(
    [("encode", OneHotEncoder(handle_unknown="ignore"), categorical_cols)],
    remainder="passthrough",  # numeric columns pass through unchanged
)

print(f"Training customers: {len(X_train):,}   Test customers: {len(X_test):,}")

## 3. Model

*Baseline first — the reference every model must beat — then the model
under study.*

The **majority-class baseline** (Week 1 Lesson 8) predicts "no churn" for
everyone. The model under study is the same random-forest pipeline Week 1
Lesson 8 used: the encoding defined in Data Prep, then the forest. The
point of this notebook isn't a better algorithm, it's evaluating the same
one honestly.

### Model Selection

Lesson 10's workflow: compare a few `max_depth` candidates by **stratified
k-fold CV on training data only** (Lesson 2), scored by **average
precision** — Lesson 4's guidance to lead with the PR metric, not accuracy
or ROC-AUC, once the positive class is a minority. The test set stays
untouched until Section 4.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

def make_model(max_depth):
    clf = RandomForestClassifier(max_depth=max_depth, random_state=RANDOM_SEED)
    return Pipeline([("prep", preprocess), ("clf", clf)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
candidates = [None, 5, 10, 20]
cv_scores = {
    depth: cross_val_score(make_model(depth), X_train, y_train, cv=cv,
                            scoring="average_precision").mean()
    for depth in candidates
}
for depth, score in cv_scores.items():
    print(f"max_depth={depth}:  avg. CV average-precision = {score:.3f}")

best_depth = max(cv_scores, key=cv_scores.get)
print(f"\nBest candidate: max_depth={best_depth}")

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
model = make_model(best_depth).fit(X_train, y_train)

## 4. Evaluate

*Score against the baseline, not in isolation — a metric only means
something next to what it's beating.*

First the accuracy paradox (Week 1 Lesson 8) made concrete on real data: the
baseline's accuracy looks respectable while catching zero churners. Then
the CV-selected model, scored once on the untouched test set, with the
precision/recall/ROC/PR toolkit from Lessons 3–4 instead of accuracy alone.

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_curve, average_precision_score,
                             roc_auc_score, roc_curve)

baseline_preds = baseline.predict(X_test)
print(f"Baseline accuracy: {accuracy_score(y_test, baseline_preds):.1%}"
      "  <- looks fine, but catches zero churners:")
# zero_division=0: the baseline never predicts "churned," so precision on
# that class is undefined by the usual 0/0 -- reporting it as 0 is the
# correct read here, not a warning-worthy edge case.
print(classification_report(y_test, baseline_preds, target_names=["stayed", "churned"],
                             zero_division=0))

In [ ]:
model_proba = model.predict_proba(X_test)[:, 1]
model_preds = model.predict(X_test)

print(f"Model accuracy: {accuracy_score(y_test, model_preds):.1%}")
print(classification_report(y_test, model_preds, target_names=["stayed", "churned"]))

tn, fp, fn, tp = confusion_matrix(y_test, model_preds).ravel()
print(f"Confusion matrix -> TP={tp}  FP={fp}  FN={fn}  TN={tn}")

roc_auc = roc_auc_score(y_test, model_proba)
ap = average_precision_score(y_test, model_proba)
print(f"ROC-AUC: {roc_auc:.3f}   Average precision (PR-AUC): {ap:.3f}"
      f"   (baseline rate: {y_test.mean():.1%})")

In [ ]:
fpr, tpr, _ = roc_curve(y_test, model_proba)
prec, rec, thresh = precision_recall_curve(y_test, model_proba)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(fpr, tpr, lw=2.5, color="#1A73E8", label=f"AUC={roc_auc:.2f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray", label="random guessing")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve"); axes[0].legend()

axes[1].plot(rec, prec, lw=2.5, color="#1A73E8", label=f"AP={ap:.2f}")
axes[1].axhline(y_test.mean(), color="gray", linestyle="--", label="random guessing")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve"); axes[1].legend()
plt.tight_layout()

## 5. Interpret

*What is the model actually keying off of? Business-readable, not just
statistically significant.*

Feature importances, same as Week 1 Lesson 8 — then the threshold question
from Lessons 4–5: at the default 0.5 cutoff the model is conservative about
flagging churn (visible in the recall above). If the retention team wants
to catch more churners, sliding the threshold down trades some precision
for recall, exactly as Lesson 5's stump demonstrated.

In [ ]:
feature_names = model.named_steps["prep"].get_feature_names_out()
importances = pd.Series(model.named_steps["clf"].feature_importances_, index=feature_names)
importances.sort_values().tail(10).plot.barh(title="Top features driving the prediction")
plt.tight_layout()

In [ ]:
# precision_recall_curve returns one more (precision, recall) pair than
# threshold -- drop the last pair so every point lines up with a threshold.
target_recall = 0.70
idx = np.argmin(np.abs(rec[:-1] - target_recall))
chosen_threshold, chosen_precision, chosen_recall = thresh[idx], prec[idx], rec[idx]
print(f"To catch ~{chosen_recall:.0%} of churners, lower the threshold to "
      f"~{chosen_threshold:.2f} (precision at that point: {chosen_precision:.2f})")

## 6. Recommend

**Recommendation:** rank customers by the model's predicted churn
probability and route the top-risk decile to the retention team, using a
threshold chosen to hit a target recall (e.g. catch ~70% of churners) rather
than the default 0.5 cutoff — the right operating point is a business cost
decision (Lesson 4), not a statistical default. The CV-selected model
clears the majority-class baseline decisively on the metric that matters
here (average precision), not just on accuracy, which the baseline could
already win by doing nothing.

**Caveats / next steps:** this notebook fixed the algorithm (random forest)
and only tuned `max_depth`, on purpose — the point was the evaluation
discipline, not the model search. Weeks 3–6 revisit this exact dataset with
interpretable and tunable model families (logistic regression, regularized
linear models, gradient boosting, neural nets), each scored with the same
metrics and CV discipline built this week.